# 🫁 Proyek ATM — Segmentasi Kanker Paru-Paru pada Citra CT-Scan

**Mata Kuliah:** Computer Vision  
**Metode:** CLAHE + Otsu Thresholding + Morfologi (OpenCV)  

---

### 📚 Referensi Jurnal
| # | Jurnal | Metode |
|---|--------|--------|
| 1 | Husni & Adrial (2023) — *Jurnal Fisika Unand* Vol.12 No.1 | Robert, Sobel, Prewitt, **Canny** |
| 2 | Fendriani dkk (2023) — *JoP* Vol.8 No.2 | Canny + **Mean/Median/Gaussian Filter** |

### 🔄 Konsep ATM
| ATM | Penjelasan |
|-----|------------|
| **Amati** | Kedua jurnal menggunakan Edge Detection pada CT-Scan kanker paru |
| **Tiru** | Studi kasus sama (CT-Scan), metrik sama (MSE & PSNR) |
| **Modifikasi** | Ganti Edge Detection → **Segmentasi** (CLAHE + Otsu + Morfologi) |

---

### 🗺️ Alur Kerja Notebook
```
CELL 1  → Mount Google Drive
CELL 2  → Install library
CELL 3  → Import library
CELL 4  → Download & eksplorasi dataset
CELL 5  → Fungsi Preprocessing (CLAHE)
CELL 6  → Fungsi Segmentasi (Otsu + Morfologi)
CELL 7  → Fungsi hitung MSE & PSNR
CELL 8  → Pipeline lengkap satu gambar
CELL 9  → Proses semua kelas
CELL 10 → Tabel ringkasan metrik
CELL 11 → Tabel perbandingan ATM (output laporan)
```

> ⚠️ **Jalankan setiap cell secara berurutan dari atas ke bawah.**

---
## 📁 CELL 1 — Mount Google Drive & Setup Folder

Menghubungkan Colab ke Google Drive agar hasil tersimpan **secara permanen**  
(tidak hilang ketika session Colab berakhir / timeout).

Folder yang akan dibuat:
```
MyDrive/
└── ATM_KankerParu/
    └── results/    ← semua output gambar & CSV disimpan di sini
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/ATM_KankerParu/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"✅ Google Drive terhubung")
print(f"📂 Folder hasil: {RESULTS_DIR}")

---
## 📦 CELL 2 — Install Library Tambahan

Hanya `kagglehub` yang perlu diinstall.  
Library lain (`cv2`, `numpy`, `matplotlib`, `pandas`) sudah tersedia di Colab.

In [ ]:
!pip install kagglehub -q
print("✅ kagglehub berhasil diinstall")

---
## 📚 CELL 3 — Import Semua Library

| Library | Fungsi |
|---------|--------|
| `cv2` (OpenCV) | Pemrosesan gambar: CLAHE, threshold, morfologi |
| `numpy` | Operasi matematika pada array pixel |
| `matplotlib` | Visualisasi gambar dan grafik |
| `os`, `glob` | Navigasi file dan folder |
| `pandas` | Membuat tabel ringkasan metrik |

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Semua library berhasil di-import")
print(f"   OpenCV versi : {cv2.__version__}")
print(f"   NumPy  versi : {np.__version__}")

---
## 📥 CELL 4 — Download & Eksplorasi Dataset

**Dataset:** `mohamedhanyyy/chest-ctscan-images` (Kaggle)

**4 Kelas yang tersedia:**
| Kelas | Keterangan |
|-------|------------|
| `adenocarcinoma` | Kanker jenis adenokarsinoma (paling umum) |
| `large.cell.carcinoma` | Kanker sel besar, pertumbuhan cepat |
| `normal` | Paru-paru sehat (tidak ada kanker) |
| `squamous.cell.carcinoma` | Kanker sel skuamosa, sering terkait rokok |

> 💡 Cell ini akan menghitung dan menampilkan jumlah gambar per kelas.

In [ ]:
import kagglehub

print("⬇️  Mengunduh dataset dari Kaggle...")
path = kagglehub.dataset_download("mohamedhanyyy/chest-ctscan-images")
print(f"✅ Dataset tersimpan di: {path}\n")

TRAIN_DIR = os.path.join(path, 'train')
TEST_DIR  = os.path.join(path, 'test')

for split_name, split_dir in [('TRAIN', TRAIN_DIR), ('TEST', TEST_DIR)]:
    print(f"📂 {split_name}:")
    total = 0
    for folder in sorted(os.listdir(split_dir)):
        folder_path = os.path.join(split_dir, folder)
        if os.path.isdir(folder_path):
            jumlah = len(os.listdir(folder_path))
            total += jumlah
            print(f"   {folder:<30} : {jumlah} gambar")
    print(f"   {'TOTAL':<30} : {total} gambar\n")

---
## ⚙️ CELL 5 — Fungsi Preprocessing (CLAHE)

### Apa itu CLAHE?
**CLAHE** = *Contrast Limited Adaptive Histogram Equalization*

Berbeda dengan histogram equalization biasa yang memproses seluruh gambar sekaligus,  
CLAHE bekerja secara **lokal** (membagi gambar menjadi grid kecil) sehingga:
- Detail di area gelap maupun terang sama-sama terlihat
- Tidak ada area yang jadi terlalu over-exposed

### Mengapa ini adalah MODIFIKASI dari jurnal?
> Jurnal referensi melakukan perbaikan kontras **manual** dengan mapping linear  
> `(low=0.2, high=0.45) → (0, 255)`.
> 
> Kita menggunakan CLAHE yang **otomatis dan adaptif** — tidak perlu menentukan nilai manual.

### Parameter CLAHE:
| Parameter | Nilai | Penjelasan |
|-----------|-------|------------|
| `clipLimit` | 2.0 | Batas amplifikasi kontras (mencegah noise berlebih) |
| `tileGridSize` | (8, 8) | Gambar dibagi menjadi grid 8×8 region |

In [ ]:
def preprocessing(img_path, img_size=256):
    """
    Baca gambar CT-Scan, ubah ke grayscale, resize,
    lalu terapkan CLAHE untuk meningkatkan kontras.

    Parameter:
        img_path (str): Path ke file gambar CT-Scan
        img_size (int): Ukuran output, default 256x256 pixel

    Return:
        gray  (ndarray): Gambar grayscale SEBELUM CLAHE
        hasil (ndarray): Gambar grayscale SESUDAH CLAHE
    """
    # Langkah 1: Baca gambar
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Gambar tidak ditemukan: {img_path}")

    # Langkah 2: Konversi BGR → Grayscale
    # CT-Scan pada dasarnya sudah grayscale, tapi file bisa
    # disimpan dalam format RGB/BGR sehingga perlu dikonversi
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Langkah 3: Resize ke ukuran seragam
    gray = cv2.resize(gray, (img_size, img_size))

    # Langkah 4: Terapkan CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    hasil = clahe.apply(gray)

    return gray, hasil


# ── Tes cepat: tampilkan contoh hasil CLAHE ──────────────────────────────────
# Ambil satu gambar dari kelas pertama sebagai demo
demo_kelas = sorted(os.listdir(TEST_DIR))[0]
demo_file  = sorted(os.listdir(os.path.join(TEST_DIR, demo_kelas)))[0]
demo_path  = os.path.join(TEST_DIR, demo_kelas, demo_file)

gray_demo, clahe_demo = preprocessing(demo_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray_demo,  cmap='gray'); axes[0].set_title('Sebelum CLAHE (Grayscale)'); axes[0].axis('off')
axes[1].imshow(clahe_demo, cmap='gray'); axes[1].set_title('Sesudah CLAHE');             axes[1].axis('off')
plt.suptitle(f'Demo CLAHE — {demo_file}', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Fungsi preprocessing siap digunakan")

---
## 🔬 CELL 6 — Fungsi Segmentasi (Otsu + Morfologi)

Ini adalah **inti dari MODIFIKASI ATM** kita.

### Perbandingan dengan Jurnal Referensi
| | Jurnal 1 & 2 | Penelitian Ini |
|-|-------------|----------------|
| **Hasil** | Garis tepi tipis (hitam-putih) | Area kanker terisi penuh |
| **Analogi** | Menggambar outline objek | Mewarnai seluruh objek |

### Tahapan Segmentasi

**1️⃣ Otsu Thresholding**  
Menentukan nilai threshold (T) secara **otomatis** dari histogram gambar.  
Pixel > T → putih (kanker), Pixel ≤ T → hitam (bukan kanker)

**2️⃣ Opening** (Erosi → Dilasi)  
Menghilangkan titik-titik putih kecil (noise) di **luar** area kanker

**3️⃣ Closing** (Dilasi → Erosi)  
Menutup lubang-lubang hitam kecil di **dalam** area kanker  
→ Hasil ini yang menjadi **output final**

In [ ]:
def segmentasi(img_clahe):
    """
    Segmentasi gambar menggunakan Otsu Thresholding + Operasi Morfologi.

    Parameter:
        img_clahe (ndarray): Gambar grayscale setelah CLAHE

    Return:
        mask_otsu    : Hasil Otsu (masih ada noise)
        mask_opening : Setelah Opening (noise luar hilang)
        mask_closing : Setelah Closing — INI HASIL FINAL
    """
    # Langkah 1: Otsu Thresholding
    # 0        = threshold awal (diabaikan, Otsu menghitung sendiri)
    # 255      = nilai untuk pixel yang lolos threshold
    # THRESH_OTSU = hitung threshold optimal otomatis
    _, mask_otsu = cv2.threshold(
        img_clahe, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Langkah 2: Buat kernel elips 5x5
    # Elips dipilih karena bentuk tumor cenderung membulat
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # Langkah 3: Opening = Erosi → Dilasi
    # Menghilangkan titik putih kecil (noise) di luar area kanker
    mask_opening = cv2.morphologyEx(mask_otsu, cv2.MORPH_OPEN, kernel)

    # Langkah 4: Closing = Dilasi → Erosi
    # Menutup lubang hitam kecil di dalam area kanker
    mask_closing = cv2.morphologyEx(mask_opening, cv2.MORPH_CLOSE, kernel)

    return mask_otsu, mask_opening, mask_closing


# ── Tes cepat: tampilkan semua tahap segmentasi ──────────────────────────────
_, clahe_demo2 = preprocessing(demo_path)
otsu_d, open_d, close_d = segmentasi(clahe_demo2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
    [clahe_demo2, otsu_d, open_d, close_d],
    ['Input (CLAHE)', 'Otsu Threshold', 'Setelah Opening', 'Setelah Closing (Final)']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Demo Tahapan Segmentasi', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Fungsi segmentasi siap digunakan")

---
## 📐 CELL 7 — Fungsi Hitung MSE & PSNR

Metrik yang **sama** dengan jurnal referensi — agar hasil bisa dibandingkan langsung.

### MSE (Mean Square Error)
$$MSE = \frac{1}{M \times N} \sum_{i,j} [I(i,j) - K(i,j)]^2$$
- Semakin **rendah** = gambar hasil makin mirip aslinya

### PSNR (Peak Signal to Noise Ratio)
$$PSNR = 10 \times \log_{10}\left(\frac{255^2}{MSE}\right)$$
- Semakin **tinggi** = kualitas gambar makin baik (lebih sedikit noise)

### Data Pembanding dari Jurnal
| Jurnal | Metode Terbaik | MSE | PSNR |
|--------|---------------|-----|------|
| Jurnal 1 | Canny | 37.278 | 2,43 dB |
| Jurnal 2 | Median Filter | 47 | 31 dB |

In [ ]:
def hitung_mse(img_asli, img_hasil):
    """
    Hitung MSE antara gambar asli dan gambar hasil segmentasi.

    Parameter:
        img_asli  : Gambar grayscale asli (sebelum proses)
        img_hasil : Gambar hasil segmentasi (mask biner)

    Return:
        float: Nilai MSE
    """
    asli  = img_asli.astype(np.float64)
    hasil = img_hasil.astype(np.float64)
    return np.mean((asli - hasil) ** 2)


def hitung_psnr(mse, max_pixel=255.0):
    """
    Hitung PSNR dari nilai MSE.

    Parameter:
        mse       : Nilai MSE yang sudah dihitung
        max_pixel : Nilai pixel maksimum (255 untuk gambar 8-bit)

    Return:
        float: Nilai PSNR dalam desibel (dB)
    """
    if mse == 0:
        return float('inf')
    return 10 * np.log10((max_pixel ** 2) / mse)


# ── Tes cepat ──────────────────────────────────────────────────────────────
gray_t, clahe_t = preprocessing(demo_path)
_, _, final_t   = segmentasi(clahe_t)

mse_t  = hitung_mse(gray_t, final_t)
psnr_t = hitung_psnr(mse_t)

print("✅ Fungsi MSE & PSNR siap digunakan")
print(f"\n   Contoh hasil pada gambar demo:")
print(f"   MSE  = {mse_t:.2f}")
print(f"   PSNR = {psnr_t:.2f} dB")
print(f"\n   Pembanding:")
print(f"   Jurnal 1 (Canny terbaik)       → MSE: 37.278 | PSNR: 2.43 dB")
print(f"   Jurnal 2 (Median Filter terbaik) → MSE: 47    | PSNR: 31 dB")

---
## 🔄 CELL 8 — Pipeline Lengkap (Proses Satu Gambar)

Fungsi ini menggabungkan semua langkah sebelumnya menjadi satu pipeline:  

```
Baca gambar
    → Grayscale + Resize
        → CLAHE
            → Otsu Thresholding
                → Opening
                    → Closing (Final)
                        → Hitung MSE & PSNR
                            → Tampilkan 5 panel visualisasi
```

Output visualisasi: **5 panel** dari gambar asli hingga hasil akhir.

In [ ]:
def proses_satu_gambar(img_path, simpan_path=None, tampilkan=True):
    """
    Pipeline lengkap: baca → preprocessing → segmentasi → metrik → visualisasi.

    Parameter:
        img_path    : Path ke file gambar CT-Scan
        simpan_path : Path untuk menyimpan visualisasi ke Drive (opsional)
        tampilkan   : True untuk menampilkan plot (default True)

    Return:
        dict: Berisi nama file dan nilai-nilai metrik
    """
    # Tahap 1: Preprocessing
    gray, clahe_img = preprocessing(img_path)

    # Tahap 2: Segmentasi
    mask_otsu, mask_opening, mask_closing = segmentasi(clahe_img)

    # Tahap 3: Hitung metrik
    mse_otsu     = hitung_mse(gray, mask_otsu)
    psnr_otsu    = hitung_psnr(mse_otsu)
    mse_closing  = hitung_mse(gray, mask_closing)
    psnr_closing = hitung_psnr(mse_closing)

    # Tahap 4: Visualisasi 5 panel
    if tampilkan:
        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        judul  = ['1. Asli (Grayscale)', '2. Setelah CLAHE',
                  '3. Otsu Threshold', '4. Setelah Opening', '5. Final (Closing)']
        gambar = [gray, clahe_img, mask_otsu, mask_opening, mask_closing]

        for ax, img, title in zip(axes, gambar, judul):
            ax.imshow(img, cmap='gray')
            ax.set_title(title, fontsize=9)
            ax.axis('off')

        nama_file = os.path.basename(img_path)
        plt.suptitle(
            f'{nama_file}\n'
            f'Otsu   → MSE: {mse_otsu:.2f}, PSNR: {psnr_otsu:.2f} dB  |  '
            f'Final  → MSE: {mse_closing:.2f}, PSNR: {psnr_closing:.2f} dB',
            fontsize=9
        )
        plt.tight_layout()

        if simpan_path:
            plt.savefig(simpan_path, dpi=150, bbox_inches='tight')

        plt.show()

    return {
        'file'       : os.path.basename(img_path),
        'mse_otsu'   : round(mse_otsu,    2),
        'psnr_otsu'  : round(psnr_otsu,   2),
        'mse_final'  : round(mse_closing,  2),
        'psnr_final' : round(psnr_closing, 2)
    }


# ── Demo: jalankan satu gambar ────────────────────────────────────────────────
print("🔬 Menjalankan demo pipeline pada satu gambar...\n")
hasil_demo = proses_satu_gambar(demo_path)
print("\n✅ Pipeline berhasil! Hasil:")
for k, v in hasil_demo.items():
    print(f"   {k:<15} : {v}")

---
## 🚀 CELL 9 — Proses Semua Kelas

Mengambil **3 sampel gambar** dari setiap kelas (4 kelas) dan memproses  
masing-masing dengan pipeline lengkap.

**Total yang diproses:** 4 kelas × 3 gambar = **12 gambar**

Semua hasil visualisasi otomatis tersimpan di **Google Drive**.

In [ ]:
KELAS = ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
HASIL_SEMUA = []

for kelas in KELAS:
    folder = os.path.join(TEST_DIR, kelas)
    if not os.path.isdir(folder):
        print(f"⚠️  Folder tidak ditemukan: {folder}")
        continue

    gambar_list = sorted(os.listdir(folder))[:3]

    print(f"\n{'='*55}")
    print(f"  📁 KELAS: {kelas.upper()}")
    print(f"{'='*55}")

    for nama_gambar in gambar_list:
        img_path = os.path.join(folder, nama_gambar)
        simpan   = os.path.join(RESULTS_DIR, f'{kelas}_{nama_gambar}.png')

        hasil = proses_satu_gambar(img_path, simpan_path=simpan, tampilkan=True)
        hasil['kelas'] = kelas
        HASIL_SEMUA.append(hasil)

        print(f"  ✔ {nama_gambar}")
        print(f"    MSE : {hasil['mse_final']}  |  PSNR : {hasil['psnr_final']} dB")

print(f"\n{'='*55}")
print(f"  ✅ Selesai! {len(HASIL_SEMUA)} gambar berhasil diproses.")
print(f"  📂 Hasil tersimpan di: {RESULTS_DIR}")
print(f"{'='*55}")

---
## 📊 CELL 10 — Tabel Ringkasan Metrik

Menampilkan semua hasil dalam bentuk tabel yang rapi dan menghitung:  
- Nilai MSE & PSNR per gambar
- Rata-rata per kelas
- Rata-rata keseluruhan

Tabel disimpan sebagai **CSV** di Google Drive untuk keperluan laporan.

In [ ]:
df = pd.DataFrame(HASIL_SEMUA)

# ── Tabel lengkap ────────────────────────────────────────────────────────────
print("=" * 70)
print("  TABEL HASIL SEGMENTASI — SEMUA GAMBAR")
print("=" * 70)
display(df[['kelas', 'file', 'mse_otsu', 'psnr_otsu', 'mse_final', 'psnr_final']]
        .rename(columns={
            'kelas'     : 'Kelas',
            'file'      : 'File',
            'mse_otsu'  : 'MSE (Otsu)',
            'psnr_otsu' : 'PSNR Otsu (dB)',
            'mse_final' : 'MSE (Final)',
            'psnr_final': 'PSNR Final (dB)'
        }))

# ── Rata-rata per kelas ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  RATA-RATA PER KELAS")
print("=" * 70)
display(df.groupby('kelas')[['mse_final', 'psnr_final']]
         .mean()
         .round(2)
         .rename(columns={'mse_final': 'MSE (Final)', 'psnr_final': 'PSNR Final (dB)'}))

# ── Rata-rata keseluruhan ─────────────────────────────────────────────────────
mse_avg  = df['mse_final'].mean()
psnr_avg = df['psnr_final'].mean()
print(f"\n{'='*70}")
print(f"  RATA-RATA KESELURUHAN")
print(f"{'='*70}")
print(f"  MSE  rata-rata : {mse_avg:.2f}")
print(f"  PSNR rata-rata : {psnr_avg:.2f} dB")

# ── Grafik MSE & PSNR per gambar ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(np.linspace(0, 1, len(KELAS)))
kelas_color = {k: c for k, c in zip(KELAS, colors)}
bar_colors = [kelas_color[row['kelas']] for _, row in df.iterrows()]

axes[0].bar(range(len(df)), df['mse_final'], color=bar_colors)
axes[0].set_title('MSE per Gambar (Final Segmentation)', fontweight='bold')
axes[0].set_xlabel('Sampel')
axes[0].set_ylabel('MSE')
axes[0].axhline(y=mse_avg, color='red', linestyle='--', label=f'Rata-rata: {mse_avg:.2f}')
axes[0].legend()
axes[0].set_xticks(range(len(df)))
axes[0].set_xticklabels([f"{row['kelas'][:4]}\n{i%3+1}" for i, (_, row) in enumerate(df.iterrows())],
                         fontsize=8)

axes[1].bar(range(len(df)), df['psnr_final'], color=bar_colors)
axes[1].set_title('PSNR per Gambar (Final Segmentation)', fontweight='bold')
axes[1].set_xlabel('Sampel')
axes[1].set_ylabel('PSNR (dB)')
axes[1].axhline(y=psnr_avg, color='red', linestyle='--', label=f'Rata-rata: {psnr_avg:.2f} dB')
axes[1].legend()
axes[1].set_xticks(range(len(df)))
axes[1].set_xticklabels([f"{row['kelas'][:4]}\n{i%3+1}" for i, (_, row) in enumerate(df.iterrows())],
                         fontsize=8)

plt.tight_layout()
grafik_path = os.path.join(RESULTS_DIR, 'grafik_mse_psnr.png')
plt.savefig(grafik_path, dpi=150, bbox_inches='tight')
plt.show()

# ── Simpan CSV ───────────────────────────────────────────────────────────────
csv_path = os.path.join(RESULTS_DIR, 'metrics_table.csv')
df.to_csv(csv_path, index=False)
print(f"\n✅ Tabel tersimpan di: {csv_path}")
print(f"✅ Grafik tersimpan di: {grafik_path}")

---
## 📋 CELL 11 — Tabel Perbandingan ATM (Output Laporan)

Tabel ini adalah **output utama untuk laporan** —  
membandingkan metode jurnal referensi dengan metode modifikasi kita.

In [ ]:
mse_avg  = df['mse_final'].mean()
psnr_avg = df['psnr_final'].mean()

# ── Tabel perbandingan ───────────────────────────────────────────────────────
data_perbandingan = {
    'Aspek'            : ['Metode', 'Preprocessing', 'Hasil Visual', 'Dataset', 'Tools', 'MSE Terbaik', 'PSNR Terbaik'],
    'Jurnal 1 (Husni)' : ['Robert, Sobel, Prewitt, Canny', 'Kontras manual (0.2-0.45)', 'Garis tepi tipis', '5 pasien RS Unand', 'MATLAB R2017b', '37.278 (Canny)', '2,43 dB'],
    'Jurnal 2 (Fendriani)': ['Canny + Mean/Median/Gaussian', 'Kontras manual', 'Garis tepi + filter', '6 pasien RS Jambi', 'MATLAB R2015a', '47 (Median Filter)', '31 dB'],
    'Penelitian Ini'   : ['CLAHE + Otsu + Morfologi', 'CLAHE (adaptif otomatis)', 'Area kanker terisi penuh', 'Kaggle CT-Scan', 'Python + OpenCV', f'{mse_avg:.2f}', f'{psnr_avg:.2f} dB'],
}

df_perbandingan = pd.DataFrame(data_perbandingan)

print("=" * 75)
print("  TABEL PERBANDINGAN ATM (AMATI - TIRU - MODIFIKASI)")
print("=" * 75)
display(df_perbandingan.set_index('Aspek'))

# ── Simpan tabel perbandingan ────────────────────────────────────────────────
perbandingan_path = os.path.join(RESULTS_DIR, 'tabel_perbandingan_ATM.csv')
df_perbandingan.to_csv(perbandingan_path, index=False)

print(f"\n✅ Tabel perbandingan tersimpan di: {perbandingan_path}")

# ── Ringkasan akhir ──────────────────────────────────────────────────────────
print(f"""
{'='*75}
  ✅ PROYEK ATM SELESAI!
{'='*75}

  Ringkasan Hasil:
  - Total gambar diproses : {len(df)}
  - MSE  rata-rata (final): {mse_avg:.2f}
  - PSNR rata-rata (final): {psnr_avg:.2f} dB

  File tersimpan di Google Drive (ATM_KankerParu/results/):
  - [kelas]_[gambar].png     → visualisasi per gambar
  - grafik_mse_psnr.png      → grafik batang MSE & PSNR
  - metrics_table.csv        → tabel metrik lengkap
  - tabel_perbandingan_ATM.csv → tabel untuk laporan
{'='*75}
""")